In [1]:
from nltk.tokenize import word_tokenize
from torch.nn import Embedding
from torch import nn
import torch
import math
input = "I love cats"
token = word_tokenize(input)


In [2]:
vocabulary = {}
vocabulary.update({ 0 :"/s"})
for i in range(0, len(token)):
    vocabulary.update({i+1 : token[i]})
vocabulary.update({4 : "EOS"})
vocab = vocabulary.__len__()  


In [3]:
# embedded = nn.Embedding(token.__len__() , embedding_dim=512)
# embedded
# embedded.weight

In [4]:
class Embedd(nn.Module):

    def __init__(self , d_model  , vocab):
        super().__init__()
        self.embedded = nn.Embedding(vocab , d_model)
        self.d_model = d_model
    def forward (self, x):
        return self.embedded(x) *  math.sqrt(self.d_model)   
    

In [5]:
class PositionalEncoding(nn.Module):
    
    def __init__(self , dropout , d_model , seq_len = 500):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(seq_len , d_model)
        position = torch.arange(0. , seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0 , d_model ,2) * - (math.log(10000.0)/d_model) )
        pe[: , 0::2] = torch.sin(position * div_term)
        pe[: , 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe' , pe)
    def forward(self , x):
        x = x + (self.pe[: , :x.shape[1] , : ]).requires_grad_(False)
        return self.dropout(x)




In [ ]:
class LayerNormalization(nn.Module) :
    def __init__(self , size , eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(size)
        self.beta = nn.Parameter(size)
        self.eps = eps
    def forward(self , x):
        mean = x.mean(-1 , keepdim=True)
        std = x.std(-1 , keepdim=True)
        return self.gammas * (x - mean) / (std + self.eps) + self.beta     

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model , d_ff , dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model , d_ff) # bias automatically is true
        self.w2 = nn.Linear(d_ff , d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self , x):
        return self.w2(self.dropout(torch.relu(self.w1(x))))        

In [ ]:
class MultiHeadAttentions(nn.Module):

    def __init__(self , d_model , head , dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.head = head
        self.dropout = nn.Dropout(p=dropout)
        assert d_model % head == 0
        self.d_k = d_model // head

        self.w_query = nn.Linear(d_model , d_model)
        self.w_key = nn.Linear(d_model , d_model)
        self.w_value = nn.Linear(d_model , d_model)
        self.v_output = nn.Linear(d_model , d_model)

    @staticmethod
    def attention(query , key , value , mask , dropout):
        d_k = query.size(-1)
        attention_score = (query @ key.transpose(-2,-1)) / math.sqrt(d_k)
        if mask is not None :
            attention_score.masked_fill_(mask == 0 , -1e9)
        attention_score = attention_score.softmax(dim = -1)
        if dropout is not None :
            attention_score = dropout(attention_score)
        return (attention_score @ value) , attention_score        
    
    def forward(self , q , k , v , mask):
        query = self.w_query(q)
        key = self.w_key(k)
        value = self.w_value(v)

        query = query.view(query.shape[0] , query.shape[1] , self.head , self.d_k).transpose(1,2)    
        key = key.view(key.shape[0] , key.shape[1] , self.head , self.d_k).transpose(1,2)
        value = value.view(value.shape[0] , value.shape[1] , self.head , self.d_k).transpose(1,2)
        x , self.attention_score = MultiHeadAttentions.attention(query , key , value , mask , self.dropout)
        x = x.transpose(1,2).contiguous().view(x.shape[0] , -1 , self.d_k * self.head)

        return self.v_output(x)


In [9]:
#Also called ResidualConnection
class LayerConnection(nn.Module):
    def __init__(self , dropout):
        super().__init__()
        self.norm = LayerNormalization()
        self.dropout = nn.Dropout(dropout)

    def forward(self , x , sublayer):
       
       return x + self.dropout(sublayer(self.norm(x)))    
           

In [ ]:
class EncoderLayer(nn.Module):

    def __init__(self , attention_block : MultiHeadAttentions , feedforwardblock : FeedForwardNetwork , dropout):
        super().__init__()
        self.attention_block = attention_block
        self.feedforwardblock = feedforwardblock
        self.residualconnection = nn.ModuleList([LayerConnection(dropout) for _ in range(2)])
    def forward(self , x , mask):
        x = self.residualconnection[0](x , lambda x : self.attention_block(x , x , x , mask))
        x= self.residualconnection[1](x , self.feedforwardblock)
        return x

In [11]:
class Encoder(nn.Module):

    def __init__(self, layers : nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()
    def forward(self , x , mask):
        for layer in self.layers :
            x = layer(x , mask)
        return self.norm(x)    
            


In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, attenntion_block : MultiHeadAttentions , feedforwardblock : FeedForwardNetwork , dropout):
        super().__init__()
        self.masked_attention_block = attenntion_block
        self.attention_block = attenntion_block
        self.feddforwardblock = feedforwardblock
        self.residualnetwork = nn.ModuleList([LayerConnection(dropout) for _ in range (2)])
    def forward(self , x , mask):
        x = self.residualnetwork[0](x , lambda x : self.masked_attention_block(x , x , x , mask ))
        x = self.residualconnection[0](x , lambda x : self.attention_block(x , x , x ))
        x= self.residualconnection[1](x , self.feedforwardblock)
        return x